## Calculate surface heat fluxes using BOWTIE data

In [1]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
## Surface exchange coefficients
### use z0 = 0.001 for the first run then 
### z0 = z0_n   # z0_n is the roughness length calculated as Equation 10 in Zeng et al. (1997)
#z0 = z0_n 
kap = 0.4
def stability_function_m(ds_air, ds_surf,z0,z1):
    """
    Calculate the stability function for the Monin-Obukhov similarity theory.
    
    Parameters:
    rib (xarray.DataArray): Richardson number.
    
    Returns:
    xarray.DataArray: Stability function.
    """
    rib = richardson_number(ds_air, ds_surf)
    z1 = z1
    z0 = z0 
    km = (kap / np.log(z1 / z0))**2
    hzh_fac = (xr.where(z1/z0>1,z1/z0,1)**(1/3) - 1)**1.5

    return np.where(rib >= 0, 1 / (1 + 10 * rib*(1 + 8*rib)), 
                     1 + 10*np.abs(rib)/(1 + 75*km*hzh_fac*(np.abs(rib))**0.5))

def stability_function_h(ds_air, ds_surf,z0,z1):
    """
    Calculate the stability function for the Monin-Obukhov similarity theory.
    
    Parameters:
    rib (xarray.DataArray): Richardson number.
    
    Returns:
    xarray.DataArray: Stability function.
    """
    rib = richardson_number(ds_air, ds_surf)
    z1 = z1
    z0 = z0
    km = (kap / np.log(z1 / z0))**2
    hzh_fac = (xr.where(z1/z0>1,z1/z0,1)**(1/3) - 1)**1.5

    return np.where(rib >= 0, 1 / (1 + 10 * rib*(1 + 8*rib)), 
                     1 + 15*np.abs(rib)/(1 + 75*km*hzh_fac*(np.abs(rib))**0.5))

def richardson_number(ds_air, ds_surf,z0,z1):
    """
    Calculate the Richardson number from potential temperature and surface temperature.
    
    Parameters:
    ds (xarray.Dataset): Dataset containing potential temperature and surface temperature.
    
    Returns:
    xarray.DataArray: Richardson number.
    """
    z1 = z1
    z0 = z0
    g = 9.81  # Acceleration due to gravity in m/s^2    
    theta = potential_temperature(ds_air)
    return (theta - ds_surf.ts) * g * (z1-z0)/ (ds_surf.ts* (ds_air.ua**2 + ds_air.va**2))

def potential_temperature(ds):
    """
    Calculate potential temperature from temperature and pressure.
    
    Parameters:
    temperature (xarray.DataArray): Temperature in Kelvin.
    pressure (xarray.DataArray): Pressure in Pascals.
    
    Returns:
    xarray.DataArray: Potential temperature in Kelvin.
    """
    Rd = 287.05  # Specific gas constant for dry air in J/(kg·K)
    cp = 1004.0  # Specific heat capacity at constant pressure in J/(
    return ds_air.ta * (101325 / ds_air.pfull) ** (Rd / cp)

def saturated_vapor_pressure(temperature):
    """
    Calculate the saturated vapor pressure from temperature.
    
    Parameters:
    temperature (xarray.DataArray): Temperature in Kelvin.
    
    Returns:
    xarray.DataArray: Saturated vapor pressure in Pascals.
    """
    return 611.2 * np.exp((17.67 * (temperature - 273.15)) / (temperature - 29.65))
def saturated_specific_humidity(ds_air, ds_surf):
    """
    Calculate specific humidity from air temperature, pressure, and relative humidity.
    
    Parameters:
    ds_air (xarray.Dataset): Dataset containing air temperature, pressure, and relative humidity.
    
    Returns:
    xarray.DataArray: Specific humidity in kg/kg.
    """
    es = saturated_vapor_pressure(ds_surf.ts)
    return 0.622 * es / (ds_air.pfull - (1-0.622)*es)

def oboukhov_length(ds_air, ds_surf,cdn,chn,z0,z1):
    """
    Calculate the Obukhov length from air temperature, pressure, and specific humidity.
    
    Parameters:
    ds_air (xarray.Dataset): Dataset containing air temperature, pressure, and specific humidity.
    
    Returns:
    xarray.DataArray: Obukhov length in meters.
    """
    z1= z1
    z0 = z0
    g = 9.81  # Acceleration due to gravity in m/s^2
    theta = potential_temperature(ds_air)
    Rv = 461.5  # Specific gas constant for water vapor in J/(kg·K)
    Rd = 287.05  # Specific gas constant for dry air in J/(kg·K)
    qs = saturated_specific_humidity(ds_air, ds_surf)
    sh = chn*(ds_air.ua**2 + ds_air.va**2)**0.5 * (ds_surf.ts-theta)
    lh = chn*(ds_air.ua**2 + ds_air.va**2)**0.5 * (qs-ds_air.hus)
    wT = sh + (Rv/Rd-1)*lh*ds_surf.ts
    ufric = (ds_air.ua**2 + ds_air.va**2)**0.5 * cdn**0.5
    L = -ufric**3 * ds_surf.ts / (g * kap *wT)
    return L, ufric

def factor_correction(ds_air, ds_surf, L, z0, z1):
    """
    Calculate the factor correction for the Monin-Obukhov similarity theory.
    
    Parameters:
    ds_air (xarray.Dataset): Dataset containing air temperature, pressure, and specific humidity.
    ds_surf (xarray.Dataset): Dataset containing surface temperature and other surface variables.
    
    Returns:
    xarray.DataArray: Factor correction.
    """
    z1= z1
    z0 = z0
    g = 9.81  # Acceleration due to gravity in m/s^2
    ##unstable
    lambda1 = ((1-16*(z1/L)))**0.25
    lambda0 = ((1-16*(z0/L)))**0.25

    #psim_u1 = 2*np.log((1 + lambda1) / 2) + np.log((1 + lambda1**2) / 2) - 2* np.arctan(lambda1) + np.pi / 2 
    #psim_u0 = 2*np.log((1 + lambda0) / 2) + np.log((1 + lambda0**2) / 2) - 2* np.arctan(lambda0) + np.pi / 2 
    psim_u1 = 2*np.log(1 + lambda1) + np.log(1 + lambda1**2) - 2* np.arctan(lambda1) + np.pi / 2 - 3*np.log(2)
    psim_u0 = 2*np.log(1 + lambda0) + np.log(1 + lambda0**2) - 2* np.arctan(lambda0) + np.pi / 2 - 3*np.log(2)

    psih_u1 = 2*np.log((1 + lambda1**2)/2)
    psih_u0 = 2*np.log((1 + lambda0**2)/2)

    f_um = (np.log(z1/z0) - psim_u1 + psim_u0)/kap
    f_uh = (np.log(z1/z0) - psih_u1 + psih_u0 )/kap

    #stable
    psim_s1 = -5*z1/L
    psim_s0 = -5*z0/L
    f_sm = (np.log(z1/z0) - psim_s1 + psim_s0)/kap
    f_sh = f_sm

    #super stable
    psim_ss1 = -5 + 5*z0 + (1-5)*np.log(z1) - z1 + 1
    f_ssm = (np.log(z1/z0) - psim_ss1) / kap
    f_ssh = f_ssm
    
    return xr.where(L>0,xr.where(z1/L>1,1/(f_ssm**2),1/(f_sm**2)),1/(f_um**2)), \
        xr.where(L>0,xr.where(z1/L>1,1/(f_ssh*f_ssm),1/(f_sh*f_sm)),1/(f_uh*f_um))

In [ ]:
rib =   richardson_number(ds_air, ds_surf)
km = (kap / np.log(12.5 / z0))**2
nu = 1.5e-5  # Kinematic viscosity in m^2/s
psim = stability_function_m(ds_air, ds_surf)
psih = stability_function_h(ds_air, ds_surf)
km_n = km*psim[0] + (rib*0)
kh_n = km*psih[0]+ (rib*0)
km_nn = km_n
kh_nn = kh_n
for t in range(5):
    Lob, ustar = oboukhov_length(ds_air, ds_surf, km_nn, kh_nn)
    cd, ch = factor_correction(ds_air, ds_surf, Lob)
    km_nn = cd
    kh_nn = ch